In [ ]:
import pandas as pd
import ssl
import socket
from datetime import datetime
from tqdm import tqdm
import re

tqdm.pandas()

from google.colab import drive
drive.mount('/content/drive')

# Load the data
df = pd.read_excel("drive/MyDrive/SSLTestNew.xls")


def get_ssl_info(hostname):
    port = 443
    if not hostname or pd.isna(hostname):
        return ""
    try:
        hostname = str(hostname).strip()
        # Remove http/https if present
        hostname = hostname.replace('http://', '').replace('https://', '')
        # Remove paths and ports
        hostname = hostname.split('/')[0].split(':')[0]

        context = ssl.create_default_context()
        # NOTE: SSLContext has no real "timeout" attribute — setting one is a
        # no-op and gives a false sense of a timeout being applied. The real
        # timeout is the one passed to socket.create_connection() below.

        with socket.create_connection((hostname, port), timeout=10) as sock:
            with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()
                return cert
    except Exception as e:
        print(f"Error getting SSL info for {hostname}: {str(e)}")
        return ""


def extract_issuer_organization(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    issuer = cert.get('issuer', {})

    # Method 1: Standard tuple format
    if isinstance(issuer, tuple):
        for field in issuer:
            for (key, value) in field:
                if key.lower() == 'organizationname':
                    return value

    # Method 2: Dictionary format
    elif isinstance(issuer, dict):
        for key, value in issuer.items():
            if 'organization' in key.lower():
                return value

    # Method 3: String parsing fallback
    issuer_str = str(issuer)
    org_match = re.search(r'O\s*=\s*([^,]+)', issuer_str)
    if org_match:
        return org_match.group(1).strip()

    # Method 4: Subject field fallback
    subject = cert.get('subject', {})
    if isinstance(subject, tuple):
        for field in subject:
            for (key, value) in field:
                if key.lower() == 'organizationname':
                    return value

    return ""


def extract_issuer_common_name(cert):
    """Extract the CA's common name (e.g. 'R3', 'DigiCert').

    Fixed: previously this ran hostname-cleanup regexes (stripping
    'www.'/'ssl.' prefixes, splitting on '.') on what is actually a
    Certificate Authority name, not a domain. That's a different kind of
    string and that cleanup logic didn't belong here. Now we just try to
    match against known CA names directly.
    """
    if not cert or not isinstance(cert, dict):
        return ""

    issuer = cert.get('issuer', {})
    common_name = ""

    # Method 1: Standard tuple format
    if isinstance(issuer, tuple):
        for field in issuer:
            for (key, value) in field:
                if key.lower() == 'commonname':
                    common_name = value
                    break

    # Method 2: Dictionary format
    elif isinstance(issuer, dict):
        for key, value in issuer.items():
            if 'commonname' in key.lower():
                common_name = value
                break

    # Method 3: String parsing fallback
    if not common_name:
        issuer_str = str(issuer)
        cn_match = re.search(r'CN\s*=\s*([^,]+)', issuer_str)
        if cn_match:
            common_name = cn_match.group(1).strip()

    if not common_name:
        return ""

    known_cas = ['Sectigo', 'DigiCert', 'GeoTrust', 'Go Daddy', 'GlobalSign', "Let's Encrypt",
                 'COMODO', 'cPanel', 'R3', 'Cloudflare', 'Amazon', 'Google', 'Microsoft']
    for ca in known_cas:
        if ca.lower() in common_name.lower():
            return ca

    # Otherwise return the CA CN as-is (no domain-style splitting/stripping)
    return common_name.strip()


def extract_wildcard_common_name(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    try:
        subject = cert.get('subject', ())

        if isinstance(subject, tuple):
            for field in subject:
                for (key, value) in field:
                    if key.lower() == 'commonname' and isinstance(value, str) and value.startswith('*.'):
                        return value

        elif isinstance(subject, dict):
            for key, value in subject.items():
                if 'commonname' in key.lower() and isinstance(value, str) and value.startswith('*.'):
                    return value

        return ""
    except Exception:
        return ""


def extract_country(cert):
    if not cert or not isinstance(cert, dict):
        return ""

    subject = cert.get('subject', {})
    country = ""

    if isinstance(subject, tuple):
        for field in subject:
            for (key, value) in field:
                if key.lower() == 'countryname':
                    country = value
                    break

    elif isinstance(subject, dict):
        for key, value in subject.items():
            if 'countryname' in key.lower():
                country = value
                break

    if not country:
        subject_str = str(subject)
        country_match = re.search(r'C\s*=\s*([^,]+)', subject_str)
        if country_match:
            country = country_match.group(1).strip()

    return country


def format_ssl_date(date_str):
    if not date_str:
        return None

    # Clean the date string
    date_str = date_str.replace('GMT', '').strip()
    # FIX: OpenSSL pads single-digit days with an extra space
    # (e.g. "Jan  6 12:00:00 2024"), which broke strptime matching below.
    # Collapse any run of whitespace down to a single space.
    date_str = re.sub(r'\s+', ' ', date_str)

    formats_to_try = [
        "%b %d %H:%M:%S %Y",
        "%Y-%m-%d %H:%M:%S",
        "%d %b %Y %H:%M:%S",
        "%Y%m%d%H%M%SZ",
        "%Y%m%d%H%M%S",
    ]

    for fmt in formats_to_try:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue

    return None


# Initialize lists to store results
results = []

# Get today's date
today_date = datetime.now().date()

# Process each domain
for domain in tqdm(df["Domain"]): # Changed 'Domain Name' to 'Domain'
    try:
        ssl_info = get_ssl_info(domain)

        if isinstance(ssl_info, dict):
            not_before = format_ssl_date(ssl_info.get("notBefore", ""))
            not_after = format_ssl_date(ssl_info.get("notAfter", ""))
            organization_name = extract_issuer_organization(ssl_info)
            issuer_common_name = extract_issuer_common_name(ssl_info)
            wildcard_cn = extract_wildcard_common_name(ssl_info)
            country = extract_country(ssl_info)

            validity_days = None
            validity_years = None
            today_str = today_date.strftime("%d/%m/%Y")
            days_remaining = None

            if not_before and not_after:
                validity_days = (not_after.date() - not_before.date()).days
                validity_years = "1 year" if validity_days < 380 else "2 years"
                days_remaining = (not_after.date() - today_date).days

            not_before_str = not_before.strftime("%d/%m/%Y") if not_before else ""
            not_after_str = not_after.strftime("%d/%m/%Y") if not_after else ""

        else:
            not_before_str, not_after_str, organization_name = "", "", ""
            issuer_common_name = ""
            validity_days, validity_years = "", ""
            today_str = today_date.strftime("%d/%m/%Y")
            wildcard_cn = ""
            days_remaining = None
            country = ""

        results.append({
            'Domain': domain,
            'SSL_Not_Before': not_before_str,
            'SSL_Not_After': not_after_str,
            'Organization_Name': organization_name,
            'Common_Name': issuer_common_name,
            'Validity_days': validity_days,
            'Validity_years': validity_years,
            'Today_date': today_str,
            'Wild_card_Common_Name': wildcard_cn,
            'days_remaining': days_remaining,
            'Country': country,
        })

    except Exception as e:
        print(f"Error processing {domain}: {str(e)}")
        results.append({
            'Domain': domain,
            'SSL_Not_Before': "",
            'SSL_Not_After': "",
            'Organization_Name': "",
            'Common_Name': "",
            'Validity_days': "",
            'Validity_years': "",
            'Today_date': today_date.strftime("%d/%m/%Y"),
            'Wild_card_Common_Name': "",
            'days_remaining': None,
            'Country': "",
        })

# Create result dataframe
result_df = pd.DataFrame(results)

# Display preview
print("\nPreview of Results:")
display(result_df.head())

# Save results
result_df.to_excel("drive/MyDrive/Lead Sample_result.xlsx", index=False)
print("\nProcessing complete. Results saved.")